# Chaos Test — AVD Session Host

**Ziel:** Den AVD Session Host gezielt unter CPU-Last setzen (statt herunterzufahren) via Azure Chaos Studio.

**Aufteilung:**
- Chaos Studio (Agent-Target + Capability + Chaos-Agent-Extension + Experiment + Rollenzuweisung) wird komplett **in Bicep** definiert und über die GitHub Action **Deploy Chaos Studio** (`infra/chaos/main.bicep`) deployed.
- Das Experiment wird **hier im Notebook mit einem einzigen Start-Call** ausgelöst, überwacht und gestoppt.

**Fault:** `urn:csci:microsoft:agent:cpuPressure/1.0` — belastet die CPU von `vm-avd-cptdazavdvwan` auf ~95 % für die Experiment-Dauer (`PT10M`). Agent-basierter Fault, benötigt den Chaos-Agent (VM-Extension) + User-Assigned Identity an der VM (beides erledigt die Action).


## Variablen

In [ ]:
export PREFIX=cptdazavdvwan
export RG=rg-${PREFIX}
export SUB=$(az account show --query id -o tsv)
export VM=vm-avd-${PREFIX}
export HP=hp-${PREFIX}
export EXPERIMENT=exp-cpu-${PREFIX}
export APIV=2024-01-01
echo "RG=$RG SUB=$SUB VM=$VM HP=$HP EXPERIMENT=$EXPERIMENT"


## 0. Voraussetzung: Chaos Studio deployed

Das Experiment muss zuvor über die GitHub Action **Deploy Chaos Studio** (`infra/chaos/main.bicep`) angelegt worden sein. Diese Zelle prüft, ob das Experiment existiert.

In [ ]:
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}?api-version=${APIV}" \
  --query "{name:name, provisioningState:properties.provisioningState, principalId:identity.principalId}" -o json

## 1. Baseline — CPU-Auslastung (vorher)

Liest die aktuelle CPU-Auslastung direkt im Gast aus (via Run Command). Erwartung: niedrige Last (Leerlauf).


In [ ]:
echo "=== CPU-Auslastung im Gast (Baseline) ==="
az vm run-command invoke -g $RG -n $VM \
  --command-id RunPowerShellScript \
  --scripts "\$c=(Get-Counter '\\Processor(_Total)\\% Processor Time').CounterSamples.CookedValue; Write-Output ('CPU Total: {0:N1} %' -f \$c); Write-Output '--- Top 5 Prozesse (CPU) ---'; Get-Process | Sort-Object CPU -Descending | Select-Object -First 5 Name,CPU | Format-Table -AutoSize | Out-String" \
  --query "value[0].message" -o tsv


## 2. Experiment starten — der eine Call

Ein einziger POST-Call startet das in Bicep definierte Experiment. Danach lesen wir die jüngste Execution aus.


In [ ]:
echo "=== Starte Experiment $EXPERIMENT ==="
az rest --method post \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/start?api-version=${APIV}" \
  -o json
echo ""
echo "Warte, bis die Execution erscheint..."
sleep 10
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/executions?api-version=${APIV}" \
  --query "reverse(sort_by(value,&properties.startedAt))[0].{id:name,status:properties.status,startedAt:properties.startedAt}" -o json

## 3. Experiment-Status überwachen

Mehrfach ausführbar. Status-Werte u.a.: `Running`, `Success`, `Failed`, `Cancelled`.

In [ ]:
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/executions?api-version=${APIV}" \
  --query "reverse(sort_by(value,&properties.startedAt))[0:3].{id:name,status:properties.status,startedAt:properties.startedAt,stoppedAt:properties.stoppedAt}" -o table

## 4. Auswirkung beobachten — CPU unter Last

Erwartung nach ~1–2 Min: CPU Total nahe dem konfigurierten `pressureLevel` (~95 %). Mehrfach ausführbar.


In [ ]:
echo "=== CPU-Auslastung im Gast (während Experiment) ==="
az vm run-command invoke -g $RG -n $VM \
  --command-id RunPowerShellScript \
  --scripts "\$c=(Get-Counter '\\Processor(_Total)\\% Processor Time').CounterSamples.CookedValue; Write-Output ('CPU Total: {0:N1} %' -f \$c); Write-Output '--- Top 5 Prozesse (CPU) ---'; Get-Process | Sort-Object CPU -Descending | Select-Object -First 5 Name,CPU | Format-Table -AutoSize | Out-String" \
  --query "value[0].message" -o tsv


## 5. Experiment vorzeitig stoppen (optional)

Bricht das laufende Experiment ab. Der Chaos-Agent beendet die CPU-Last, die Auslastung normalisiert sich danach von selbst.


In [ ]:
az rest --method post \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/cancel?api-version=${APIV}" \
  -o json
echo "Cancel ausgelöst." 

## 6. Recovery prüfen

Nach Ablauf der Dauer (oder nach Cancel) lässt die CPU-Last nach. Diese Zelle prüft, dass die Auslastung wieder im Leerlauf-Bereich liegt.


In [ ]:
echo "=== CPU-Auslastung im Gast (Recovery) ==="
az vm run-command invoke -g $RG -n $VM \
  --command-id RunPowerShellScript \
  --scripts "\$c=(Get-Counter '\\Processor(_Total)\\% Processor Time').CounterSamples.CookedValue; Write-Output ('CPU Total: {0:N1} %' -f \$c)" \
  --query "value[0].message" -o tsv
